In [ ]:
_repo_root = !git rev-parse --show-toplevel
%cd {_repo_root[0]}
del _repo_root

# HIGGS feature-sensitivity bounds

Publication figures for the empirical deviation ratios of trained HIGGS spectral neurons.

## Setup

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from paper.experiments.higgs_robustness import (
    default_result_path,
    validate_results,
)
from paper.experiments.higgs_scaling import PROFILES
from paper.plotting import plot_higgs_deviation_shell_grid

mpl.rcParams.update({
    "figure.dpi": 140,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
profile_name = "full"
noise_level = 0.5
shell_count = 4
profile = PROFILES[profile_name]
result_path = default_result_path(profile_name, noise_level)
plot_dir = Path("plots")

## Method

The experiment stores a fine joint histogram over perturbation magnitude and deviation ratio. The figure merges adjacent magnitude bins into disjoint shells, normalizes within each trained model and shell, and then gives every model equal weight. The ratio range is `[0, 1]`; ratios above the theoretical bound are not clipped and remain visible in the diagnostics below.

## Load and validate results

In [ ]:
if not result_path.exists():
    raise FileNotFoundError(
        f"{result_path} does not exist; run paper.experiments.higgs_robustness first"
    )

results = pd.read_csv(result_path)
validate_results(results, profile, noise_level=noise_level)
print(result_path)

## Bound diagnostics

In [ ]:
histogram_columns = [
    "dim", "data_seed", "init_seed", "perturbation_seed",
    "noise_level", "feature_index", "magnitude_bin_index",
    "total_count",
    "zero_bound_count", "above_bound_count", "max_ratio",
]
histograms = results[histogram_columns]
diagnostics = (
    histograms.groupby(["noise_level", "dim"], as_index=False)
    .agg(
        histogram_cells=("magnitude_bin_index", "size"),
        total_ratios=("total_count", "sum"),
        zero_bounds=("zero_bound_count", "sum"),
        above_bound=("above_bound_count", "sum"),
        max_ratio=("max_ratio", "max"),
    )
)
diagnostics

## Deviation-ratio distributions

In [ ]:
plot_dir.mkdir(parents=True, exist_ok=True)
figure = plot_higgs_deviation_shell_grid(results, shell_count=shell_count)
noise_label = f"{noise_level:g}".replace(".", "p")
stem = plot_dir / (
    f"higgs_robustness_{profile_name}_noise_{noise_label}_shells_{shell_count}"
)
figure.savefig(f"{stem}.pdf", bbox_inches="tight")
figure.savefig(f"{stem}.png", dpi=300, bbox_inches="tight")
display(figure)
plt.close(figure)